# BigSmall Multitask Training Notebook

This notebook trains the BigSmall multitask model for rPPG, respiration, and AU estimation.

In [ ]:
import os
import sys
import json
import glob
import pickle
import shutil
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import torch
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

REPO_ROOT = "d:/New folder/Non-Invasive/rPPG"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from neural_methods.model.BigSmall import BigSmall


In [ ]:
# ----- paths -----
RAW_DATA_PATH       = os.path.join(REPO_ROOT, "data/Headmotion")
PREPROCESSED_PATH   = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/bigsmall")

MODEL_SAVE_PATH     = os.path.join(REPO_ROOT, "final_model_release", "BigSmall_Multitask.pth")
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

# ----- video / signal params -----
VIDEO_FPS   = 30       # camera frame rate
PPG_FS      = 60       # PPG sensor sampling rate (Hz)

# ----- BigSmall preprocessing params -----
CHUNK_LENGTH = 3       # frames per clip (same as training config)
BIG_H, BIG_W = 144, 144
SMALL_H, SMALL_W = 9, 9

# ----- device -----
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ----- 49-channel label layout (matches BigSmallTrainer label_list) -----
LABEL_IDX_BVP  = 0   # bp_wave (green-channel PPG signal)
LABEL_IDX_HR   = 1   # HR_bpm
LABEL_IDX_RESP = 5   # resp_wave
NUM_LABEL_CH   = 49

os.makedirs(PREPROCESSED_PATH, exist_ok=True)
print("PREPROCESSED_PATH:", PREPROCESSED_PATH)
print("MODEL_SAVE_PATH:", MODEL_SAVE_PATH)


## 1. Preprocessing Tools (Optional)

These functions can be used to convert raw video and `.csv` ground truths into BigSmall's required multi-scale format.

In [ ]:
# Read video frames
def read_video_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()

    if not frames:
        raise ValueError(f"Empty video: {video_path}")
    return np.stack(frames, axis=0)


In [ ]:
def read_ppg_synced(session_path, num_frames):
    import pandas as pd
    import numpy as np
    import os
    
    frame_df = pd.read_csv(os.path.join(session_path, "frame_timestamps.csv"))
    
    if len(frame_df) != num_frames:
        min_len = min(len(frame_df), num_frames)
        frame_t = frame_df["timestamp"].values[:min_len]
    else:
        frame_t = frame_df["timestamp"].values

    ppg_df = pd.read_csv(os.path.join(session_path, "ppg.csv"))
    ppg_t = ppg_df["Timestamp"].values
    ppg_val = ppg_df["PPG"].values
    
    frame_t_clipped = np.clip(frame_t, ppg_t[0], ppg_t[-1])
    ppg_resampled = np.interp(frame_t_clipped, ppg_t, ppg_val)
    
    return ppg_resampled.astype(np.float32)


In [ ]:
# Normalization functions
def diff_normalize_data(data):
    data = data.astype(np.float32)
    n, h, w, c = data.shape
    out = np.zeros_like(data)
    out[:n - 1] = (data[1:] - data[:-1]) / (data[1:] + data[:-1] + 1e-7)
    std = np.std(out)
    if std > 0:
        out /= std
    return out

def standardized_data(data):
    data = data.astype(np.float32)
    m = np.mean(data)
    s = np.std(data)
    if s > 0:
        data = (data - m) / s
    else:
        data = np.zeros_like(data)
    data = np.where(np.isnan(data), np.zeros_like(data), data)
    return data

def diff_normalize_label(label):
    diff = np.diff(label.astype(np.float64), axis=0)
    s = np.std(diff)
    if s > 0:
        diff = diff / s
    return np.append(diff, [0.0]).astype(np.float32)


In [ ]:
# Face crop + resize
def crop_face_resize(frames, out_h, out_w, large_box_coef=1.5):
    xml_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    detector  = cv2.CascadeClassifier(xml_path)

    frame0 = frames[0]
    if frame0.dtype != np.uint8:
        frame0 = np.clip(frame0, 0, 255).astype(np.uint8)
    gray = cv2.cvtColor(frame0, cv2.COLOR_RGB2GRAY)

    faces = detector.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    H, W = frames.shape[1], frames.shape[2]

    if len(faces) > 0:
        x, y, fw, fh = max(faces, key=lambda f: f[2])  # largest face
        x  = max(0, int(x  - (large_box_coef - 1.0) / 2.0 * fw))
        y  = max(0, int(y  - (large_box_coef - 1.0) / 2.0 * fh))
        fw = min(int(fw * large_box_coef), W - x)
        fh = min(int(fh * large_box_coef), H - y)
    else:
        x, y, fw, fh = 0, 0, W, H  # fallback: full frame

    C = frames.shape[3]
    resized = np.zeros((len(frames), out_h, out_w, C), dtype=np.float32)
    for i, frame in enumerate(frames):
        crop = frame[y : y + fh, x : x + fw]
        if crop.size == 0:
            crop = frame
        resized[i] = cv2.resize(crop.astype(np.float32), (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized

def resize_frames(frames, out_h, out_w):
    T, H, W, C = frames.shape
    resized = np.zeros((T, out_h, out_w, C), dtype=np.float32)
    for i in range(T):
        resized[i] = cv2.resize(frames[i], (out_w, out_h),
                                interpolation=cv2.INTER_AREA)
    return resized


In [ ]:
# Discover subjects and preprocess (Set PREPROCESS_DATA = True to run)
all_dirs = sorted([
    d for d in glob.glob(os.path.join(RAW_DATA_PATH, "*"))
    if os.path.isdir(d) and os.path.basename(d) != "videos"
])
subjects = []
for subj_dir in all_dirs:
    subj_id  = os.path.basename(subj_dir)
    subj_key = subj_id.replace("_", "")
    video_pattern = os.path.join(RAW_DATA_PATH, "videos", f"{subj_id}.mkv")
    video_files = glob.glob(video_pattern)
    if not video_files:
        continue
    subjects.append({
        "subj_id":      subj_id,
        "subj_key":     subj_key,
        "video_path":   video_files[0],
        "session_path": subj_dir,
    })

# Set PREPROCESS_DATA = False if data is already prepared in PREPROCESSED_PATH
PREPROCESS_DATA = False 
if PREPROCESS_DATA:
    if os.path.exists(PREPROCESSED_PATH):
        shutil.rmtree(PREPROCESSED_PATH)
    os.makedirs(PREPROCESSED_PATH)

    for subj in subjects:
        subj_key     = subj["subj_key"]
        video_path   = subj["video_path"]
        session_path = subj["session_path"]
        
        frames = read_video_frames(video_path)
        T = frames.shape[0]
        ppg_signal = read_ppg_synced(session_path, T)

        labels = np.full((T, NUM_LABEL_CH), -1.0, dtype=np.float32)
        labels[:, LABEL_IDX_BVP] = ppg_signal
        
        frames_big = crop_face_resize(frames, BIG_H, BIG_W)
        big_data   = standardized_data(frames_big)
        diff_big   = diff_normalize_data(frames_big)
        small_data = resize_frames(diff_big, SMALL_H, SMALL_W)
        labels[:, LABEL_IDX_BVP] = diff_normalize_label(ppg_signal)

        clip_num    = T // CHUNK_LENGTH
        big_clips   = np.array([big_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]   for i in range(clip_num)])
        small_clips = np.array([small_data[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH] for i in range(clip_num)])
        label_clips = np.array([labels[i*CHUNK_LENGTH:(i+1)*CHUNK_LENGTH]     for i in range(clip_num)])

        subj_dir = os.path.join(PREPROCESSED_PATH, subj_key)
        os.makedirs(subj_dir, exist_ok=True)

        for chunk_idx in range(clip_num):
            input_path = os.path.join(subj_dir, f"{subj_key}_input{chunk_idx}.pickle")
            label_path = os.path.join(subj_dir, f"{subj_key}_label{chunk_idx}.npy")
            frames_dict = {0: big_clips[chunk_idx], 1: small_clips[chunk_idx]}
            with open(input_path, "wb") as fh:
                pickle.dump(frames_dict, fh, protocol=pickle.HIGHEST_PROTOCOL)
            np.save(label_path, label_clips[chunk_idx])
            print(f"Processed {subj_key} - Chunk {chunk_idx}")


## 2. Dataset and DataLoader

In [ ]:
# Gather all files for DataLoader
all_input_files = []
for subj_dir in glob.glob(os.path.join(PREPROCESSED_PATH, "*")):
    if os.path.isdir(subj_dir):
        subj_files = glob.glob(os.path.join(subj_dir, "*_input*.pickle"))
        all_input_files.extend(subj_files)
print(f"Found {len(all_input_files)} preprocessed clips.")

class BigSmallDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label").replace(".pickle", ".npy")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        with open(self.inputs[index], "rb") as fh:
            data = pickle.load(fh)

        # NDHWC -> NDCHW
        data[0] = np.float32(np.transpose(data[0], (0, 3, 1, 2)))
        data[1] = np.float32(np.transpose(data[1], (0, 3, 1, 2)))

        label = np.float32(np.load(self.labels[index]))  # (D, 49)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id

def bigsmall_collate(batch):
    data_dicts, labels, subjects, chunk_ids = zip(*batch)

    data_big   = torch.stack([torch.from_numpy(d[0]) for d in data_dicts], dim=0)
    data_small = torch.stack([torch.from_numpy(d[1]) for d in data_dicts], dim=0)
    labels_t   = torch.stack([torch.from_numpy(l)    for l in labels],     dim=0)

    return data_big, data_small, labels_t, list(subjects), list(chunk_ids)

dataset = BigSmallDataset(all_input_files)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True,
                          num_workers=4, collate_fn=bigsmall_collate)
print(f"DataLoader ready: {len(train_loader)} batches")


## 3. Model, Loss, Optimizer

In [ ]:
# Training Setup
model = BigSmall(n_segment=CHUNK_LENGTH).to(DEVICE)

AU_weights = torch.as_tensor([9.64, 11.74, 16.77, 1.05, 0.53, 0.56, 
                              0.75, 0.69, 8.51, 6.94, 5.03, 25.00]).to(DEVICE)

criterionAU = nn.BCEWithLogitsLoss(pos_weight=AU_weights).to(DEVICE)
criterionBVP = nn.MSELoss().to(DEVICE)
criterionRESP = nn.MSELoss().to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0)

# 49 channel list layout from trainer
label_list = ['bp_wave', 'HR_bpm', 'systolic_bp', 'diastolic_bp', 'mean_bp', 
              'resp_wave', 'resp_bpm', 'eda', 
              'AU01', 'AU02', 'AU04', 'AU05', 'AU06', 'AU06int', 'AU07', 'AU09', 'AU10', 'AU10int', 
              'AU11', 'AU12', 'AU12int', 'AU13', 'AU14', 'AU14int', 'AU15', 'AU16', 'AU17', 'AU17int', 
              'AU18', 'AU19', 'AU20', 'AU22', 'AU23', 'AU24', 'AU27', 'AU28', 'AU29', 'AU30', 'AU31', 
              'AU32', 'AU33', 'AU34', 'AU35', 'AU36', 'AU37', 'AU38', 'AU39',
              'pos_bvp','pos_env_norm_bvp']

au_labels = ['AU01', 'AU02', 'AU04', 'AU06', 'AU07', 'AU10', 'AU12', 'AU14', 'AU15', 'AU17', 'AU23', 'AU24']
LABEL_IDXS_AU = [label_list.index(l) for l in au_labels]

BASE_LEN = CHUNK_LENGTH


## 4. Inline Training Loop

In [ ]:
# Training Loop
EPOCHS = 10
print("Starting Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in tbar:
        data_big, data_small, labels_t, batch_subjects, batch_chunk_ids = batch
        
        N, D, C_b, H_b, W_b = data_big.shape
        N, D, C_s, H_s, W_s = data_small.shape
        
        # Flatten temporal dimension
        big_flat   = data_big.view(N * D, C_b, H_b, W_b).to(DEVICE)
        small_flat = data_small.view(N * D, C_s, H_s, W_s).to(DEVICE)
        
        # Trim to multiple of BASE_LEN for TSM alignment
        trim = (N * D) // BASE_LEN * BASE_LEN
        big_flat   = big_flat[:trim]
        small_flat = small_flat[:trim]
        
        labels_flat = labels_t.view(N * D, NUM_LABEL_CH).to(DEVICE)
        labels_flat = labels_flat[:trim]
        
        optimizer.zero_grad()
        
        # Forward pass
        au_out, bvp_out, resp_out = model((big_flat, small_flat))
        
        # BVP Loss
        target_bvp = labels_flat[:, LABEL_IDX_BVP].unsqueeze(-1)
        bvp_loss = criterionBVP(bvp_out, target_bvp)
        
        # RESP Loss
        target_resp = labels_flat[:, LABEL_IDX_RESP].unsqueeze(-1)
        resp_loss = criterionRESP(resp_out, target_resp)
        
        # AU Loss
        target_au = labels_flat[:, LABEL_IDXS_AU]
        target_au = torch.clamp(target_au, 0.0, 1.0) # Clamp to 0-1 for BCEWithLogitsLoss
        au_loss = criterionAU(au_out, target_au)
        
        loss = bvp_loss + resp_loss + au_loss
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        tbar.set_postfix(loss=f"{loss.item():.4f}")

    if len(train_loader) > 0:
        print(f"Epoch {epoch+1} finished. Avg Loss: {running_loss/len(train_loader):.4f}")

print("Training Complete!")
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")
